# 01 — Molecular Descriptor Matrix Preparation

## AI-driven QSAR of δ-Opioid Receptor Ligands

This notebook documents the **descriptor-integration stage** that precedes the machine-learning workflow in the companion notebook `02_DOR_AI_QSAR_GitHub_Implementation.ipynb`.

The original research workflow combined descriptor families generated from multiple cheminformatics sources into a single molecular descriptor matrix (`all_descs.csv`). This notebook presents that workflow in a clean, reproducible form.

### Workflow

**Individual descriptor tables → identifier/target alignment → horizontal concatenation → quality-control checks → `all_descs.csv` → ML/QSAR workflow**

> **Portfolio note:** Only data and descriptor files that you are permitted to redistribute should be placed in the public repository. The notebook itself documents the computational workflow without requiring the publication's full dataset to be publicly redistributed.

## 1. Imports and configuration

The original notebook contained exploratory imports from several scikit-learn modules. For this preprocessing stage, only the libraries required for data integration and quality control are retained.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Repository paths
DATA_DIR = Path("data/raw")
OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)

## 2. Descriptor sources

The original workflow assembled the descriptor matrix from the following descriptor families:

- Basak descriptors
- BCUT descriptors
- Constitutional descriptors
- Geary autocorrelation descriptors
- Moran autocorrelation descriptors
- Moreau–Broto autocorrelation descriptors
- CATS2D descriptors
- Charge descriptors
- Connectivity descriptors
- E-State descriptors
- Topological descriptors
- MOE descriptors
- 3D autocorrelation descriptors
- CPSA descriptors
- RDF descriptors
- Petitjean shape-index descriptors
- WHIM descriptors

The source notebook also used refined versions of several 2D descriptor tables.

In [ ]:
# Map the descriptor family to the expected input filename.
# Update filenames if your local repository uses different names.

DESCRIPTOR_FILES = {
    "basak": "df_basak_descriptors_refined.csv",
    "bcut": "df_bcut_descriptors_refined.csv",
    "constitution": "df_constitution_descriptors_refined.csv",
    "geary": "df_geary_descriptors_refined.csv",
    "moran": "df_moran_descriptors_refined.csv",
    "moreau_broto": "df_moreaubroto_descriptors_refined.csv",
    "cats2d": "molecules_feature_CATS2D_pybiomed_refined.csv",
    "charge": "molecules_feature_charge_pybiomed_refined.csv",
    "connectivity": "molecules_feature_connectivity_pybiomed_refined.csv",
    "estate": "molecules_feature_estate_pybiomed_refined.csv",
    "topology": "molecules_feature_topology_pybiomed_refined.csv",
    "moe": "molecules_MOE_descriptors.csv",
    "autocorrelation_3d": "autocorrelation_3d.csv",
    "cpsa_3d": "CPSA_3d.csv",
    "rdf_3d": "RDF_3d.csv",
    "petitjean_3d": "petitjeanshapeindex_3d.csv",
    "whim_3d": "WHIM-3d.csv",
}

pd.DataFrame({
    "Descriptor family": DESCRIPTOR_FILES.keys(),
    "Input file": DESCRIPTOR_FILES.values()
})

## 3. Load descriptor tables

The original notebook loaded the individual descriptor tables separately and then combined them horizontally.

The helper below keeps that operation explicit and makes it easy to identify missing files.

In [ ]:
def load_table(name, filename):
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Missing descriptor file for '{name}': {path}"
        )
    table = pd.read_csv(path)
    print(f"{name:20s} {table.shape}")
    return table

descriptor_tables = {
    name: load_table(name, filename)
    for name, filename in DESCRIPTOR_FILES.items()
}

## 4. Inspect identifiers and targets

The source workflow treats `ChEMBL ID` and `pKi` as key non-descriptor fields. Some 3D descriptor tables use `Name` instead of `ChEMBL ID`.

Before concatenation, inspect these fields to confirm that the descriptor tables correspond to the same compound ordering or alignment scheme used in the study.

In [ ]:
for name, table in descriptor_tables.items():
    print(f"\n{name}")
    print("Columns:", list(table.columns[:10]))
    print("Shape:", table.shape)

    id_candidates = [
        c for c in ["ChEMBL ID", "Canonical_Smiles", "SMILES", "Name", "pKi"]
        if c in table.columns
    ]
    print("Identifier/target candidates:", id_candidates)

## 5. Standardize the descriptor tables for concatenation

For the final matrix, identifier/target columns should not be duplicated across descriptor blocks.

The original workflow removed:

- `ChEMBL ID` and `pKi` from descriptor tables where present.
- `Canonical_Smiles` and selected common physicochemical fields from the MOE table.
- `Name` from the 3D descriptor tables.

The target `pKi` and the primary compound identifier are retained from the first aligned table.

In [ ]:
def descriptor_only(table, drop_columns):
    cols = [c for c in drop_columns if c in table.columns]
    return table.drop(columns=cols).copy()

# Preserve the identifiers/target from the first descriptor table.
base = descriptor_tables["basak"].copy()

# Drop duplicated identifiers/target from additional descriptor blocks.
cleaned = {}

for name, table in descriptor_tables.items():
    if name == "basak":
        cleaned[name] = table.copy()
    elif name == "moe":
        cleaned[name] = descriptor_only(
            table,
            ["ChEMBL ID", "Canonical_Smiles", "MW", "LogP",
             "NumHDonors", "NumHAcceptors", "pKi"]
        )
    elif name in {"autocorrelation_3d", "cpsa_3d", "rdf_3d",
                  "petitjean_3d", "whim_3d"}:
        cleaned[name] = descriptor_only(table, ["Name"])
    else:
        cleaned[name] = descriptor_only(table, ["ChEMBL ID", "pKi"])

for name, table in cleaned.items():
    print(f"{name:20s} {table.shape}")

## 6. Check row alignment before horizontal concatenation

The original workflow used `pd.concat(..., axis=1, join="inner")`, which assumes that rows correspond across descriptor tables.

For a portfolio-quality workflow, this assumption should be made explicit and checked before creating the final matrix.

If your descriptor tables have a common molecular identifier, use it for a stronger alignment check. If the 3D tables contain only `Name`, verify the correspondence during data preparation before running this notebook.

In [ ]:
row_counts = pd.Series({
    name: len(table) for name, table in cleaned.items()
}).sort_values()

display(row_counts.to_frame("Number of rows"))

if row_counts.nunique() == 1:
    print("All descriptor tables have the same number of rows.")
else:
    print("WARNING: descriptor tables have different row counts.")
    print("The final inner join will retain only rows shared across tables.")

## 7. Combine descriptor blocks

The source workflow horizontally concatenated all descriptor tables using an inner join. The same operation is implemented below, while keeping the process transparent.

In [ ]:
ordered_names = list(cleaned.keys())

all_descs = pd.concat(
    [cleaned[name] for name in ordered_names],
    axis=1,
    join="inner"
)

print("Combined descriptor matrix:", all_descs.shape)
display(all_descs.head())

## 8. Check duplicate descriptor names

Different descriptor generators can occasionally produce overlapping column names. Duplicate names can cause ambiguity in downstream feature selection and model interpretation.

We therefore report them explicitly rather than silently modifying the published workflow.

In [ ]:
duplicate_columns = all_descs.columns[all_descs.columns.duplicated()].tolist()

print("Number of duplicated column names:", len(duplicate_columns))

if duplicate_columns:
    print("Duplicated names:")
    print(duplicate_columns[:50])
else:
    print("No duplicated descriptor names detected.")

## 9. Basic matrix quality control

In [ ]:
print("Matrix dimensions:", all_descs.shape)
print("Missing values:", int(all_descs.isna().sum().sum()))
print("Duplicate rows:", int(all_descs.duplicated().sum()))

if "pKi" in all_descs.columns:
    print("pKi missing values:", int(all_descs["pKi"].isna().sum()))
    print("pKi summary:")
    display(all_descs["pKi"].describe())

if "ChEMBL ID" in all_descs.columns:
    print("Unique ChEMBL IDs:", all_descs["ChEMBL ID"].nunique())

## 10. Separate identifiers, target and descriptors

The resulting matrix is organized into:

- `ChEMBL ID` — compound identifier
- `pKi` — biological activity target
- remaining columns — molecular descriptors

This is the exact hand-off required by the companion QSAR notebook.

In [ ]:
if "ChEMBL ID" in all_descs.columns:
    compound_id = all_descs["ChEMBL ID"]
else:
    compound_id = pd.Series(np.arange(len(all_descs)), name="Compound_ID")

if "pKi" not in all_descs.columns:
    raise ValueError("The final matrix does not contain the required pKi target.")

X_descriptors = all_descs.drop(columns=[c for c in ["ChEMBL ID", "pKi"] if c in all_descs.columns])
y = all_descs["pKi"]

print("Descriptor matrix X:", X_descriptors.shape)
print("Target y:", y.shape)

## 11. Export `all_descs.csv`

The resulting file is the input to:

`02_DOR_AI_QSAR_GitHub_Implementation.ipynb`

For the public repository, use a location such as `data/processed/all_descs.csv` and confirm that the underlying data are permitted to be redistributed.

In [ ]:
output_path = OUTPUT_DIR / "all_descs.csv"
all_descs.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Shape: {all_descs.shape}")